In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb
import sqlalchemy
import sklearn

print("✅ pandas:", pd.__version__)
print("✅ numpy:", np.__version__)
print("✅ xgboost:", xgb.__version__)
print("✅ sqlalchemy:", sqlalchemy.__version__)
print("✅ sklearn:", sklearn.__version__)

✅ pandas: 3.0.1
✅ numpy: 2.4.3
✅ xgboost: 3.2.0
✅ sqlalchemy: 2.0.48
✅ sklearn: 1.8.0


In [21]:
data = {
    'donor_id':        [1,    2,    3,    4,    5   ],
    'accepted_count':  [8,    2,    5,    0,    10  ],
    'total_responses': [10,   10,   10,   5,    10  ],
    'days_since_last': [3,    90,   15,   999,  1   ],
}

df = pd.DataFrame(data)
print(df)

cold_start = df[df['total_responses'] < 5]
print("Cold Start Donors:")
print(cold_start)

   donor_id  accepted_count  total_responses  days_since_last
0         1               8               10                3
1         2               2               10               90
2         3               5               10               15
3         4               0                5              999
4         5              10               10                1
Cold Start Donors:
Empty DataFrame
Columns: [donor_id, accepted_count, total_responses, days_since_last]
Index: []


In [ ]:
df['acceptance_rate'] = df['accepted_count'] / df['total_responses'].clip(lower=1)
df['recency_score']   = np.exp(-df['days_since_last'] / 60)

print(df[['donor_id', 'acceptance_rate', 'recency_score']])

# 1. إضافة عمود جديد بناءً على شرط
df['is_active'] = df['days_since_last'] < 30
print(df[['donor_id', 'days_since_last', 'is_active']])

In [29]:
# 2. تصنيف المتبرع (بدون loop)
import numpy as np

df['tier'] = np.where(
    df['acceptance_rate'] >= 0.7, 'Gold',
    np.where(
        df['acceptance_rate'] >= 0.4, 'Silver',
        'Bronze'
    )
)
print(df[['donor_id', 'acceptance_rate', 'tier']])

   donor_id  acceptance_rate    tier
0         1              0.8    Gold
1         2              0.2  Bronze
2         3              0.5  Silver
3         4              0.0  Bronze
4         5              1.0    Gold


In [33]:
# 3. تطبيق دالة على عمود كامل
def score_formula(row):
    return (row['acceptance_rate'] * 0.5) + (row['recency_score'] * 0.3)

# ✅ apply() — أسرع من loop
df['rule_based_score'] = df.apply(score_formula, axis=1)

df['rule_based_score_v2'] = (
    (df['acceptance_rate'] * 0.50) +
    (df['recency_score']   * 0.30) +
    (df['accepted_count'] / 10).clip(0, 1) * 0.20
)
print(df[['donor_id', 'rule_based_score', 'rule_based_score_v2']])

   donor_id  rule_based_score  rule_based_score_v2
0         1      6.853688e-01         8.453688e-01
1         2      1.669390e-01         2.069390e-01
2         3      4.836402e-01         5.836402e-01
3         4      1.762455e-08         1.762455e-08
4         5      7.950414e-01         9.950414e-01


In [7]:
from sklearn.model_selection import train_test_split

X = df[['acceptance_rate', 'recency_score']]
y = pd.Series([1, 0, 1, 0, 1])

model = xgb.XGBClassifier(n_estimators=10, max_depth=2)
model.fit(X, y)

print("✅ Model trained successfully")

✅ Model trained successfully


In [8]:
new_donor = pd.DataFrame({
    'acceptance_rate': [0.80],
    'recency_score':   [0.95],
})

probability = model.predict_proba(new_donor)[0][1]
print(f"Acceptance Probability: {probability:.2%}")

Acceptance Probability: 60.00%
